In [ ]:
"""
Example script to run the electricity model.
It consists of three parts:
1. Generation
2. Grid
3. Storage
"""

import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
import warnings
from pint.errors import UnitStrippedWarning
warnings.simplefilter("ignore", UnitStrippedWarning)

from imagematerials.electricity.constants import STANDARD_SCEN_EXTERNAL_DATA

from imagematerials.model import (
    GenericMaterials,
    GenericStocks,
    MaterialIntensities,
    SharesInflowStocks,
    ElectricVehicleBatteries,
    EvBatteryLinkModule
)
from imagematerials.factory import ModelFactory, Sector
from imagematerials.preprocessing import get_preprocessing_data
import prism


# TODO absolute path of file "preprocessing.py" ? current solution can differ depending on IDE used
path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials
path_base = Path(path_base, "data", "raw")

YEAR_START = 1971   # start year of the simulation period
YEAR_END = 2100     # end year of the calculations
YEAR_OUT = 2100     # year of output generation = last year of reporting


# Combined run - all electricity subsectors (no V2G)

In [ ]:
scenario_list = {"SSP2_baseline":("SSP2_baseline", None),
                # "SSP2_narrow":("SSP2_narrow", "narrow"),
                 }

# scenario_base_path = Path(path_base) / 'circular_economy_scenarios'

time_start = 2000
time_end = 2100
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
    # climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
    circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA


    elc_sector = get_preprocessing_data("electricity", path_base,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False, standard_scenario = scenario) 
    # elc_sector is a list of preprocessing data for each electricity subsector
    
    factory = ModelFactory(
    elc_sector, complete_timeline
    ).add(GenericStocks, ["elc_gen",
                         "elc_grid_lines",
                         "elc_grid_add",
                         "elc_stor_phs"
                          ]
    ).add(SharesInflowStocks, ["elc_stor_other"],
    ).add(MaterialIntensities, ["elc_gen",
                         "elc_grid_lines",
                         "elc_grid_add",
                         "elc_stor_phs",
                         "elc_stor_other"
                          ]
    )
    model = factory.finish()
    
    model.simulate(simulation_timeline)
    all_output[scen_id] = model

    print(f"Finished {scen_id}")
    # list(model.elc_stor_other)
    
output_no_v2g = all_output.copy()


# Combined run 2 - all electricity subsectors + V2G

In [ ]:
scenario_list = {"SSP2_baseline":("SSP2_baseline", None),
                # "SSP2_narrow":("SSP2_narrow", "narrow"),
                 }

# scenario_base_path = Path(path_base) / 'circular_economy_scenarios'

time_start = 2000
time_end = 2100
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
    # climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
    circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA

    vhc_sector = get_preprocessing_data("vehicles", path_base, 
                                    climate_policy_scenario_dir = climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None)
    ev_battery_sector = get_preprocessing_data("ev_battery", path_base,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False, standard_scenario = scenario)     
    elc_sector = get_preprocessing_data("electricity", path_base,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False, standard_scenario = scenario) 
    # elc_sector is a list of preprocessing data for each electricity subsector
    

    factory = ModelFactory(
        [vhc_sector, ev_battery_sector, *elc_sector], complete_timeline
        ).add(GenericStocks, ["vehicles",
                              "elc_gen",
                              "elc_grid_lines",
                              "elc_grid_add",
                              "elc_stor_phs"]
        # ).add(GenericMaterials, ["vehicles"]
        ).add(ElectricVehicleBatteries, ["ev_battery"], input_sources={
        "stock_by_cohort": "vehicles",
        "inflow": "vehicles",
        "outflow_by_cohort": "vehicles"}
        ).add(EvBatteryLinkModule, ["elc_stor_other"], input_sources={
        "stock_battery_kWh_v2g": "ev_battery"}
        ).add(SharesInflowStocks, ["elc_stor_other"],
        ).add(MaterialIntensities, ["elc_gen",
                                    "elc_grid_lines",
                                    "elc_grid_add",
                                    "elc_stor_phs",
                                    "elc_stor_other"
                                    ]
        )
    model = factory.finish()

    model.simulate(simulation_timeline)
    all_output[scen_id] = model
    
    print(f"Finished {scen_id}")

output_with_v2g = all_output.copy()
list(model.elc_stor_other)


## Example Plot

In [ ]:
model1 = output_no_v2g["SSP2_baseline"]
model2 = output_with_v2g["SSP2_baseline"]

# Panel 1: inflow
var_str1 = "inflow"
var1_1 = model1.elc_stor_other[var_str1].to_array().sum(["Region", "Type"])
var2_1 = model2.elc_stor_other[var_str1].to_array().sum(["Region", "Type"])

# Panel 2: stock_by_cohort
var_str2 = "stock_by_cohort"
var1_2 = model1.elc_stor_other[var_str2].sum(["Region", "Cohort", "Type"])
var2_2 = model2.elc_stor_other[var_str2].sum(["Region", "Cohort", "Type"])

# Panel 3: outflow_by_cohort
var_str3 = "outflow_by_cohort"
var1_3 = model1.elc_stor_other[var_str3].to_array().sum(["Region", "Cohort", "Type"])
var2_3 = model2.elc_stor_other[var_str3].to_array().sum(["Region", "Cohort", "Type"])

# --- Figure with two subplots ---
fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharex=False)

# --- Left panel: inflow ---
axes[0].plot(var1_1.time, var1_1, label="No V2G")
axes[0].plot(var2_1.time, var2_1, label="With V2G")
axes[0].set_xlabel("Year")
axes[0].set_ylabel(f"{var_str1} dedicated storage (kWh)")
axes[0].set_title(f"{var_str1} dedicated storage")
axes[0].grid(alpha=0.3)

# --- Middle panel: stock_by_cohort ---
axes[1].plot(var1_2.Time, var1_2, label="No V2G")
axes[1].plot(var2_2.Time, var2_2, label="With V2G")
axes[1].set_xlabel("Year")
axes[1].set_ylabel(f"{var_str2} dedicated storage (kWh)")
axes[1].set_title(f"{var_str2} dedicated storage")
axes[1].grid(alpha=0.3)
axes[1].set_xlim(2000, None)  # Set x-axis limit to start from 0

# --- Right panel: outflow_by_cohort ---
axes[2].plot(var1_3.time, var1_3, label="No V2G")
axes[2].plot(var2_3.time, var2_3, label="With V2G")
axes[2].set_xlabel("Year")
axes[2].set_ylabel(f"{var_str3} dedicated storage (kWh)")
axes[2].set_title(f"{var_str3} dedicated storage")
axes[2].grid(alpha=0.3)

# --- Shared legend below both plots ---
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center",
           bbox_to_anchor=(0.5, -0.05), ncol=2,
           fontsize=12, frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
var = model.elc_stor_other["inflow_materials"].to_array().sum(["Type", "Region"])

fig, ax = plt.subplots(figsize=(11,8))
for mat in var.material.values:
    plt.plot(var.time, var.sel(material=mat), label=str(mat))
plt.xlabel("Year")
plt.ylabel("Inflow of materials)")
plt.grid(alpha=0.3)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3, fontsize=12, frameon=False)

plt.title("Inflow of materials in dedicated storage")

# Generation -------------------------------------------------------

In [ ]:
scenario_list = {#"SSP2_baseline":("SSP2_baseline", None),
                #  "SSP2_narrow":("SSP2_narrow", ["narrow"]),
                "SSP2_slow":("SSP2_slow", ["slow"]),
                 }

time_start = 1971
complete_timeline = prism.Timeline(time_start, 2030, 1) #YEAR_END
simulation_timeline = prism.Timeline(YEAR_START, 2030, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    
    # Paths to scenario data
    print(climate_scen)
    print(circular_scen)
    if circular_scen is not None:
        climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
        circular_economy_scenario_dirs = {
            scenario: Path(path_base, "circular_economy_scenarios", scenario) for scenario in circular_scen # path to circular economy scenario data
        }
    else:
        climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
        circular_economy_scenario_dirs = None
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA

    elc_sector = get_preprocessing_data("electricity", path_base,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dirs, cache = False) 
    # elc_sector is a list of preprocessing data for each electricity subsector

    elc_sector_gen = next(item for item in elc_sector if item.name == "elc_gen")

    factory = ModelFactory(
        elc_sector_gen, complete_timeline
        ).add(GenericStocks
        ).add(MaterialIntensities
        )
    model = factory.finish()

    model.simulate(simulation_timeline)
    print(f"Finished {scen_id}")
    list(model.elc_gen)

# Grid -------------------------------------------------------------

## Lines ----

In [ ]:
scenario_list = {#"SSP2_baseline":("SSP2_baseline", None),
                "SSP2_narrow":("SSP2_narrow", ["narrow"]),
                 }

time_start = 1971
complete_timeline = prism.Timeline(time_start, 2030, 1) #YEAR_END
simulation_timeline = prism.Timeline(YEAR_START, 2030, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    
    # Paths to scenario data
    print(climate_scen)
    print(circular_scen)
    if circular_scen is not None:
        climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
        circular_economy_scenario_dirs = {
            scenario: Path(path_base, "circular_economy_scenarios", scenario) for scenario in circular_scen # path to circular economy scenario data
        }
    else:
        climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
        circular_economy_scenario_dirs = None
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA

    elc_sector = get_preprocessing_data("electricity", path_base,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dirs, cache = False) 
    # elc_sector is a list of preprocessing data for each electricity subsector

    elc_sector_grid_lines = next(item for item in elc_sector if item.name == "elc_grid_lines")

    factory = ModelFactory(
        elc_sector_grid_lines, complete_timeline
        ).add(GenericStocks
        ).add(MaterialIntensities
        )
    model_lines = factory.finish()

    model_lines.simulate(simulation_timeline)
    print(f"Finished {scen_id}")
    list(model_lines.elc_grid_lines)

## Grid Additions ----

In [ ]:
scenario_list = {#"SSP2_baseline":("SSP2_baseline", None),
                #"SSP2_narrow":("SSP2_narrow", ["narrow"]),
                "SSP2_slow":("SSP2_slow", ["slow","narrow"]),
                 }

time_start = 1971
complete_timeline = prism.Timeline(time_start, 2030, 1) #YEAR_END
simulation_timeline = prism.Timeline(YEAR_START, 2030, 1)

all_output = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    
    # Paths to scenario data
    print(climate_scen)
    print(circular_scen)
    if circular_scen is not None:
        climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
        circular_economy_scenario_dirs = {
            scenario: Path(path_base, "circular_economy_scenarios", scenario) for scenario in circular_scen # path to circular economy scenario data
        }
    else:
        climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
        circular_economy_scenario_dirs = None
    scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA

    elc_sector = get_preprocessing_data("electricity", path_base,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dirs, cache = False) 
    # elc_sector is a list of preprocessing data for each electricity subsector

    elc_sector_grid_add = next(item for item in elc_sector if item.name == "elc_grid_add")

    factory = ModelFactory(
        elc_sector_grid_add, complete_timeline
        ).add(GenericStocks
        ).add(MaterialIntensities
        )
    model_add = factory.finish()

    model_add.simulate(simulation_timeline)
    print(f"Finished {scen_id}")
    list(model_add.elc_grid_add)

# Storage -------------------------------------------------------

In [ ]:
climate_scen = "SSP2_baseline"
circular_scen = None

time_start = 1971
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_END, 1)

climate_policy_scenario_dir = Path(path_base, "image", climate_scen)
# climate_policy_scenario_dir = Path(path_base, "IMAGE_CircoMod", climate_scen)
circular_economy_scenario_dir = None #Path(path_base, "circular_economy_scenarios", circular_scen)
scenario = path_base /"electricity"/ STANDARD_SCEN_EXTERNAL_DATA


elc_sector = get_preprocessing_data("electricity", path_base, climate_policy_scenario_dir, circular_economy_scenario_dir, cache = False) 
# elc_sector is a list of preprocessing data for each electricity subsector

In [ ]:
# Pumped Hydropower Storage (PHS) =====

elc_sector_stor_phs = next(item for item in elc_sector if item.name == "elc_stor_phs")

factory = ModelFactory(
    elc_sector_stor_phs, complete_timeline
    ).add(GenericStocks
    ).add(MaterialIntensities
    )
model_phs = factory.finish()

model_phs.simulate(simulation_timeline)
list(model_phs.elc_stor_phs)

In [ ]:
# Other Storage =====

elc_sector_stor_other = next(item for item in elc_sector if item.name == "elc_stor_other")
# sec_electr_stor_oth = Sector("electr_stor_oth", prep_data_oth_storage, check_coordinates=False)

factory = ModelFactory(
    elc_sector_stor_other, complete_timeline
    ).add(SharesInflowStocks
    ).add(MaterialIntensities
    )
model_other = factory.finish()

model_other.simulate(simulation_timeline)
list(model_other.elc_stor_other)